# Task 3 soft mixture baseline on Kaggle
Attach the extracted task2-delivery bundle as a private dataset. Run setup, smoke check, then the baseline. Never use smoke weights as final evidence. Save backups before ending the session. Full defaults: 2 gate-only epochs + 8 joint epochs. Optuna comes later.


In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from pathlib import Path
import subprocess, sys
repo = Path('/kaggle/working/GenAI_A1')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AaizUrRehmanKazmi/GenAI_A1.git', str(repo)], check=True)
os.chdir(repo)
# For a NEW search, ensure these new files were pushed. For resume, keep the same revision.
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)
assert (repo / 'training/train_classifier.py').is_file()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-optimization.txt'], check=True)


If installation requests a restart, restart and rerun setup. For later resume, keep the same source revision; do not blindly pull changed training sources.


In [ ]:
import torch
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Enable a GPU in notebook settings'
assert torch.cuda.device_count() == 1, 'Restart and run setup before importing torch'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


In [ ]:
raw = repo / 'data/raw/oxford_pets'
raw.mkdir(parents=True, exist_ok=True)
for name in ['images', 'annotations']:
    if not (raw / name).is_dir():
        subprocess.run(['wget', '-c', '-P', str(raw), f'https://thor.robots.ox.ac.uk/pets/{name}.tar.gz'], check=True)
        subprocess.run(['tar', '-xzf', str(raw / f'{name}.tar.gz'), '-C', str(raw)], check=True)
subprocess.run([sys.executable, '-m', 'scripts.prepare_pets'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.generate_manifests'], check=True)


In [ ]:
# Locate the complete selected Task 2 bundle (four named model folders).
candidates = [p.parent for p in Path('/kaggle/input').rglob('manifest.json')
              if all((p.parent / name / 'best.pt').is_file() for name in ['classifier','salt','blur','occlusion'])]
print(candidates)
assert len(candidates) == 1, 'Attach one extracted task2-delivery bundle, or set bundle to its exact folder.'
bundle = candidates[0]


In [ ]:
# Tiny smoke check: 2 source images, 10 validation cases, 1 completed epoch.
smoke = Path('/kaggle/working/task3-smoke')
cmd = [sys.executable, '-u', '-m', 'training.train_task3_moe', '--bundle', str(bundle),
       '--output-dir', str(smoke), '--device', 'cuda', '--train-limit', '2', '--val-images', '1',
       '--stop-after-epoch', '1', '--max-hours', '0.25']
if (smoke / 'last.pt').exists(): cmd += ['--resume', str(smoke / 'last.pt')]
subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
# New baseline: leave restore_folder=None. After a reset attach the FULL Task 3 run backup.
import shutil
output = Path('/kaggle/working/task3-baseline')
restore_folder = None  # exact extracted folder containing last.pt, not the Task 2 bundle
if restore_folder is not None and not output.exists():
    source = Path(restore_folder)
    assert (source / 'last.pt').is_file()
    shutil.copytree(source, output)
cmd = [sys.executable, '-u', '-m', 'training.train_task3_moe', '--bundle', str(bundle),
       '--output-dir', str(output), '--device', 'cuda', '--max-hours', '2']
if (output / 'last.pt').exists(): cmd += ['--resume', str(output / 'last.pt')]
subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
import json
from IPython.display import FileLink, display
history = json.loads((output / 'history.json').read_text())
if history:
    best = min(history, key=lambda row: row['val_loss'])
    print('Completed:', history[-1]['epoch'], 'Best epoch:', best['epoch'], 'Stage:', best['stage'])
    print('Fixed-alpha validation loss:', best['val_loss'], 'L1:', best['val_l1'], 'SSIM:', best['val_ssim'])
    print('Per-condition/severity metrics and weights:', best['validation']['by_condition_severity'])
backup = shutil.make_archive('/kaggle/working/task3-baseline_backup', 'zip', output.parent, output.name)
display(FileLink(backup))


Repeat the baseline cell to resume toward 10 TOTAL epochs. Keep config, source revision, data and one visible GPU unchanged. The time budget is soft, not a runtime guarantee. Download the entire backup even if training paused mid-epoch. Model selection uses condition-balanced reconstruction loss with fixed alpha 0.8; training loss includes CE and balance and is not directly comparable.
